# Analyse FAIRness


In [ ]:
from earthcode.fairtool import add_fairtool_results_to_product
import pystac
from pathlib import Path
import os
import json
from earthcode.fairtool import product_audit_to_fair_dict, analyse_product
import pystac
pystac.set_stac_version('1.0.0')

In [ ]:
# from earthcode.fairtool import score_catalog
# score_catalog()

### 1. Collect the FAIR data

In [ ]:
FAIR_METRICS = {
    "fair:Findable_has_doi": "Findable: Has DOI",
    "fair:Findable_rich_metadata": "Findable: Rich metadata",
    "fair:Findable_identifier": "Findable: Identifier",
    "fair:Findable_stac_assets": "Findable: STAC assets",
    "fair:Findable_indexed": "Findable: Indexed",
    "fair:Findable_indexed_approved_metadata": "Findable: Approved metadata host",
    "fair:Findable_indexed_approved_data": "Findable: Approved data host",
    "fair:Accessible_general": "Accessible: General access",
    "fair:Accessible_protocols": "Accessible: Open protocols",
    "fair:Accessible_files": "Accessible: Readable files",
    "fair:Interoperable_uses_formal_language": "Interoperable: Formal language",
    "fair:Interoperable_controlled_vocabularies": "Interoperable: Controlled vocabularies",
    "fair:Interoperable_related_links": "Interoperable: Related links",
    "fair:Interoperable_has_documentation": "Interoperable: Documentation",
    "fair:Reusable_rich_descriptions": "Reusable: Rich descriptions",
    "fair:Reusable_has_license": "Reusable: License",
    "fair:Reusable_workflow_exists": "Reusable: Workflow provenance",
    "fair:Reusable_cloud_assets_rate": "Reusable: Cloud-native assets",
    "fair:Reusable_has_visualisation": "Reusable: Visualisation",
    "fair:Reusable_has_access_example": "Reusable: Access example",
}

PRINCIPLE_COLORS = {
    "Findable": "#2563eb",
    "Accessible": "#0f766e",
    "Interoperable": "#7c3aed",
    "Reusable": "#b45309",
}

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

catalog = Path("../open-science-catalog-metadata/products")
data = pd.DataFrame([json.loads(p.read_text()) for p in sorted(catalog.glob("*/collection.json"))])
fair = data[list(FAIR_METRICS)]
total = len(data)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120, "figure.constrained_layout.use": True,
    "font.size": 11, "axes.titlesize": 16, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 18,
    "axes.edgecolor": "white", "axes.grid": False,
    "text.color": "#1e293b", "ytick.color": "#475569",
})
print(f"Collected {total} products and {fair.shape[1]} FAIR metrics.")

In [ ]:
data[(data['fair:Reusable_cloud_assets_rate'] == 1.0)].id.values

### 2. FAIR metrics — number / total

In [ ]:
counts = fair.gt(0).sum()
counts = counts[counts < total]
labels = [FAIR_METRICS[key] for key in counts.index]
colours = [PRINCIPLE_COLORS[label.split(":")[0]] for label in labels]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(labels, [total] * len(counts), color="#f1f5f9", height=0.6)
ax.barh(labels, counts, color=colours, height=0.6)
for row, count in enumerate(counts):
    ax.text(total * 1.03, row, f"{count} / {total}", va="center")
ax.invert_yaxis()
ax.set(title="FAIR metrics · below full coverage", xlim=(0, total * 1.21), xticks=[])
plt.show()

### 3. Dataset visualisation options

Counts are **products, not files**, and categories can overlap. The viewer count is printed below the chart using the existing FAIR result. Zarr includes all Zarr assets, regardless of item validation status. Parquet requires geometry or latitude/longitude metadata.

All locally linked items are read. Where there are no local items, the linked STAC catalogs are inspected using **samples of up to five items and three child branches per catalog** (one API page, no pagination). Remote counts reflect formats found in those samples, not an exhaustive inventory. Data files are not downloaded or tested.

“Access link only” has no item or child links. “Missing or unavailable metadata” combines failed/empty STAC lookups with products that have neither item metadata nor an access link. Product IDs are in `groups`; failed URLs are in `metadata_errors`.

In [ ]:
import requests
from functools import cache
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor

metadata_errors = []

@cache
def read_stac(url):
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()

def external_items(url, seen=None):
    seen = set() if seen is None else seen
    if url in seen:
        return []
    seen.add(url)
    try:
        document = read_stac(url)
    except (requests.RequestException, ValueError) as error:
        metadata_errors.append({"url": url, "error": str(error)})
        return []
    if document.get("type") == "Feature":
        return [document]
    if document.get("type") == "FeatureCollection":
        return document.get("features", [])[:5]
    items = []
    # Sample remote catalogs: five items and three child branches per catalog.
    for relation, limit in [("item", 5), ("items", 1), ("child", 3)]:
        links = [link for link in document.get("links", []) if link["rel"] == relation]
        for link in links[:limit]:
            href = urljoin(url, link["href"])
            if relation == "items":
                href += ("&" if "?" in href else "?") + "limit=5"
            items.extend(external_items(href, seen))
    return items

In [ ]:
# Inspect remote catalogs only where the OSC has no local item links.
external_urls = sorted({link["href"] for links in data["links"]
    if not any(link["rel"] == "item" for link in links)
    for link in links if link["rel"] == "child"})
with ThreadPoolExecutor(max_workers=8) as pool:
    remote_items = dict(zip(external_urls, pool.map(external_items, external_urls)))

groups = {name: set() for name in [
    "COG / GeoTIFF", "Zarr", "Parquet + spatial metadata",
    "Other item files (NetCDF, ZIP, etc.)", "Access link only",
    "Missing or unavailable metadata",
]}

for path in sorted(catalog.glob("*/collection.json")):
    product = json.loads(path.read_text())
    product_id = product["id"]
    links = product.get("links", [])
    items = [json.loads((path.parent / link["href"]).read_text()) for link in links if link["rel"] == "item"]
    children = [link["href"] for link in links if link["rel"] == "child"]
    if not items:
        items = [item for url in children for item in remote_items.get(url, [])]
    if not items:
        if children:
            category = "Missing or unavailable metadata"
        elif any(link["rel"] == "via" and "access" in link.get("title", "").lower() for link in links):
            category = "Access link only"
        else:
            category = "Missing or unavailable metadata"
        groups[category].add(product_id)

    for item in items:
        for asset in item.get("assets", {}).values():
            if set(asset.get("roles", [])) & {"metadata", "thumbnail", "overview"}:
                continue
            file_type = (asset.get("type", "") + " " + asset.get("href", "")).lower()
            category = "Other item files (NetCDF, ZIP, etc.)"
            if "tiff" in file_type or ".tif" in file_type or "image/cog" in file_type:
                category = "COG / GeoTIFF"
            elif "zarr" in file_type:
                category = "Zarr"
            elif "parquet" in file_type:
                properties = item.get("properties", {})
                columns = asset.get("parquet:columns") or properties.get("parquet:columns") or properties.get("table:columns", [])
                names = {name.lower() for name in columns} if isinstance(columns, dict) else {column["name"].lower() for column in columns}
                if "geoparquet" in file_type or any("geometry" in name for name in names) or {"latitude", "longitude"} <= names or {"lat", "lon"} <= names:
                    category = "Parquet + spatial metadata"
            groups[category].add(product_id)

counts = pd.Series({name: len(ids) for name, ids in groups.items()})

In [ ]:
groups['Parquet + spatial metadata']

In [ ]:
ax = counts.plot.barh(figsize=(12, 6), color="#0f766e", width=0.6)
ax.bar_label(ax.containers[0], labels=[f"{n} / {total}" for n in counts], padding=8)
ax.invert_yaxis()
ax.set(title="Dataset visualisation options", xlim=(0, total * 1.21), xticks=[], ylabel="")
plt.show()

print(f"Products with viewers: {fair['fair:Reusable_has_visualisation'].eq(True).sum()} out of {total} products")
# print("External catalogs:", *external_urls, sep="\n")

### 4. Licenses — proprietary vs not proprietary


In [ ]:
licenses = data["license"].str.strip().fillna("Missing")
proprietary = licenses.eq("proprietary")
counts = pd.Series({"Proprietary": proprietary.sum(), "Not proprietary": (~proprietary).sum()})

print(f"FAIR license flag: {fair['fair:Reusable_has_license'].eq(True).sum()} / {total} marked True")
ax = counts.plot.barh(figsize=(9, 2.8), color=["#b45309", "#2563eb"], width=0.55)
ax.bar_label(ax.containers[0], labels=[f"{n} / {total}" for n in counts], padding=8)
ax.invert_yaxis()
ax.set(title="Actual license values · proprietary vs not proprietary", xlim=(0, total * 1.21), xticks=[], ylabel="")
plt.show()

display(licenses.value_counts().rename_axis("License").to_frame("Products"))
display(data.loc[proprietary, ["id", "license"]].reset_index(drop=True))

### 5. All metrics

In [ ]:
from matplotlib.patches import Patch

scores = fair.mean()
fig, ax = plt.subplots(figsize=(13, 7))
wedges, _ = ax.pie([1] * len(scores), startangle=90, counterclock=False,
    colors=[PRINCIPLE_COLORS[label.split(":")[0]] for label in FAIR_METRICS.values()],
    wedgeprops={"width": 0.45, "edgecolor": "white"})
for wedge, score in zip(wedges, scores):
    wedge.set_alpha(0.15 + 0.85 * score)
ax.text(0, 0.06, f"{scores.mean():.1%}", ha="center", va="center", fontsize=44, weight="bold")
ax.text(0, -0.17, "overall mean", ha="center", fontsize=16)

legend = []
for principle, colour in PRINCIPLE_COLORS.items():
    score = scores.filter(like=f"fair:{principle}_").mean()
    legend.append(Patch(color=colour, label=f"{principle}  {score:.1%}"))
ax.legend(handles=legend, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False,
    fontsize=24, handlelength=2.2, handleheight=1.2, labelspacing=1.0)
ax.set_title("Overall FAIR assessment", fontsize=24)
plt.show()